# TEC-Magnetometer Co-localization Analysis
## May 11, 2024 Extreme Geomagnetic Storm Event
================================================================================
### **OBJECTIVE:**
  Correlate DASI magnetometer data with GNSS TEC enhancements during the 
  May 11, 2024 auroral breakup. Students will learn to:
  
  1. Query the Madrigal database for real magnetometer data using both MadrigalWeb and HAPI
  2. Process distributed magnetometer observations (H, D, Z components)
  3. Visualize multi-station magnetometer signatures
  4. Map spatial relationships to auroral TEC enhancements
  5. Interpret field-aligned currents and ionospheric coupling
  6. Identify timing correlations between geomagnetic and ionospheric signatures

### **REFERENCE:**
  Foster, J. C., et al. (2024). Imaging the May 2024 extreme aurora with 
  ionospheric total electron content. Geophysical Research Letters, 51, 
  e2024GL111981. https://doi.org/10.1029/2024GL111981

### **DATA SOURCE:**
  MIT Haystack Observatory [Madrigal Database](https://cedar.openmadrigal.org)
  CPI MAGSTAR DASI (Distributed Arrays of Small Instruments) [Magnetometer Network](http://magstar.cpi.com/)

In [1]:
USE_HAPI = True # CHANGE ME to use MadrigalWeb or Madrigal-HAPI server
MADRIGAL_URL = "https://cedar.openmadrigal.org/"

In [2]:
# Imports + setup; extra santity checks for Helio

import datetime
import importlib
import os, os.path
import traceback
import subprocess
import sys

NEEDED_IMPORTS = [
    "hapiclient",
    "h5py",
    "matplotlib",
    "madrigalWeb.madrigalWeb",
    "numpy",
    "pandas"
]

OPTIONAL_IMPORTS = [
    "cartopy"
]

HAS_CARTOPY = False

for this_module in NEEDED_IMPORTS:
    try:
        # sanity check
        importlib.import_module(this_module)
    except:
        try:
            if this_module == "madrigalWeb.madrigalWeb":
                ret = os.system(f"pip install madrigalWeb")
            else:
                ret = os.system(f"pip install {this_module}")
            if ret != 0:
                raise
        except:
            traceback.print_exc()
            print(f"Could not import module {this_module}")

for this_module in OPTIONAL_IMPORTS:
    try:
        # sanity check
        importlib.import_module(this_module)
        HAS_CARTOPY = True
    except:
        try:
            ret = os.system(f"pip install {this_module}")
            if ret != 0:
                pass
        except:
            traceback.print_exc()
            print(f"Could not import optional module {this_module}")

# this should all work now
from hapiclient import hapi
import h5py
import matplotlib.pyplot
import madrigalWeb.madrigalWeb
import numpy
import pandas
import cartopy
#matplotlib.use('TkAgg')

In [3]:
# Event parameters + station metadata

EVENT_DATE = "2024-05-11"
DAY_AFTER = "2024-05-12"
EVENT_START_UT = "02:00:00Z"  # Event onset
EVENT_END_UT = "02:20:00Z"    # Extended window (20 minutes)
BREAKUP_PEAK_UT = "02:06:00Z" # Peak auroral breakup (visible in all data sources)

# DASI station metadata sourced from 
# Madrigal instrument metadata: https://cedar.openmadrigal.org/instMetadata
# Madrigal kind of data metadata: https://cedar.openmadrigal.org/kindatMetadata
DASI_STATIONS = {
    'DASI-VA': {
        'kinst': 8260,
        'kindat': 1060,
        'latitude': 38.7,
        'longitude': -77.8,
        'altitude_km': 0.0,
        'location': 'Staunton, Virginia (Blue Ridge)'
    },
    'DASI-HAYSTACK': {
        'kinst': 8265,
        'kindat': 1060,
        'latitude': 42.6,
        'longitude': -71.5,
        'altitude_km': 0.0,
        'location': 'Westford, Massachusetts (Haystack Observatory)'
    },
    'DASI-NEWBRITAIN': {
        'kinst': 8268,
        'kindat': 1060,
        'latitude': 41.6,
        'longitude': -72.7,
        'altitude_km': 0.0,
        'location': 'New Britain, Connecticut'
    },
    'DASI-HENNEPIN':{
        'kinst':8266,
        'kindat': 1060,
        'latitude':44.9,
        'longitude':-93.7,
        'altitude_km':0.0,
        'location': 'Hennepin, Minnesota'
    },
    'DASI-SUGARHILLS':{
         'kinst':8271,
        'kindat': 1060,
         'latitude':47.1,
         'longitude':-93.7,
         'altitude_km':0.0,
         'location': 'Minnesota'
     }
}

# Define colors to plot each station
colors = {
    'DASI-VA': "#E53421",           # Red
    'DASI-HAYSTACK': '#3498DB',     # Blue
    'DASI-NEWBRITAIN': '#2ECC71',    # Green
    'DASI-HENNEPIN': "#C62AA6",    # Magenta
    'DASI-SUGARHILLS': "#061093",    # Dark blue
    'DASI-AUGUSTA': "#DC9513",    # Orange
}

# Key Auroral & TEC Features (from Foster et al. 2024)
AURORAL_FEATURES = {
    'Breakup_Center': {
        'lat': 44.0,
        'lon': -94.0,
        'description': 'Auroral breakup center (TEC mapping)',
        'marker_style': {'marker': '*', 'size': 300, 'color': 'red'}
    },
    'Missouri_Skies_ASC': {
        'lat': 40.25,
        'lon': -94.32,
        'description': 'Missouri Skies all-sky camera (clear skies)',
        'marker_style': {'marker': 'D', 'size': 100, 'color': 'orange'}
    },
    'St_Bonifacius_MN': {
        'lat': 44.9,
        'lon': -93.7,
        'description': 'Midwest Magstar chain (peak at 02:06 UT)',
        'marker_style': {'marker': '^', 'size': 100, 'color': 'purple'}
    }
}

In [ ]:
# Download and plot GPS/GNSS TEC data from 
# Madrigal with magnetometer location overlay

# Convert to datetime objects
event_start = datetime.datetime.strptime(f"{EVENT_DATE} {EVENT_START_UT}", "%Y-%m-%d %H:%M:%S%z")
event_end = datetime.datetime.strptime(f"{EVENT_DATE} {EVENT_END_UT}", "%Y-%m-%d %H:%M:%S%z")

# *UNVERIFIED* user ID info
user_fullname = 'CEDAR Example'
user_email = 'example@gmail.com'
user_affiliation= 'CEDAR/GEM tutorial day 2026'

# World-wide GNSS Receiver Network kinst obtained from https://cedar.openmadrigal.org/instMetadata
gnss_kinst = 8000
# World-wide GNSS Receiver Network kindat obtained from https://cedar.openmadrigal.org/kindatMetadata
gnss_kindat = 3500

# output data file
tectmpfile = "/tmp/tec.txt"

# avoid re-downloading data again...
if os.access(tectmpfile, os.R_OK):
    pass
else:
    # globalIsprint.py command generated by https://cedar.openmadrigal.org/downloadAdvancedScript
    globalIsprintCmd = f"globalIsprint.py --verbose --url={MADRIGAL_URL} --parms=UT1_UNIX,GDLAT,GLON,TEC --output={tectmpfile} --user_fullname=\"{user_fullname.split()[0]}+{user_fullname.split()[1]}\" --user_email={user_email} --user_affiliation=\"{"".join([u + "+" for u in user_affiliation.split()])}\" --startDate=05/11/2024 --endDate=05/12/2024 --inst={gnss_kinst} --kindat={gnss_kindat}"
    print(f"running globalisrpint cmd {globalIsprintCmd}")

    # run command and capture result 
    result = subprocess.run(globalIsprintCmd.split(), capture_output=True, text=True)
    output_string = result.stdout
    err = result.stderr

# use pandas to parse result file
df = pandas.read_csv(tectmpfile, sep=r'\s+')
df["UT1_UNIX"] = pandas.to_datetime(df["UT1_UNIX"], unit='s', utc=True)
filtered_df = df[df["UT1_UNIX"].between(event_start, event_end)]
tec_lon = filtered_df["GLON"]
tec_lat = filtered_df["GDLAT"]
tec = filtered_df["TEC"]

# Generate plot
if HAS_CARTOPY:
    # create map projection
    fig = matplotlib.pyplot.figure()
    ax = fig.add_subplot(111, projection=cartopy.crs.PlateCarree())
    # Add map features
    ax.coastlines(resolution='50m', linewidth=0.5)
    ax.add_feature(cartopy.feature.STATES, linestyle=':', linewidth=0.5)
    ax.gridlines(draw_labels=False, linestyle='--', alpha=0.3)

    # Plot 20 minute TEC scatter
    tecmap = ax.scatter(tec_lon, tec_lat, c=tec, cmap='jet', transform=cartopy.crs.PlateCarree(),vmin=0.0,vmax=60.0, label='GNSS TEC')
    # Add the TEC color legend
    fig.colorbar(tecmap, label='TECu', shrink=0.4)

    # Plot stations
    for station_name, station_info in DASI_STATIONS.items():
        ax.scatter(station_info['longitude'], station_info['latitude'],
                   s=150, marker='s', color=colors[station_name],
                   edgecolors='black', linewidths=1.5,
                   transform=cartopy.crs.PlateCarree(), zorder=5,
                   label=station_name)
    
    # Plot auroral features
    for feature_name, feature in AURORAL_FEATURES.items():
        marker = feature['marker_style']['marker']
        size = feature['marker_style']['size']
        color = feature['marker_style']['color']
        
        ax.scatter(feature['lon'], feature['lat'],
                   s=size, marker=marker, color=color,
                   edgecolors='black', linewidths=1,
                   transform=cartopy.crs.PlateCarree(), zorder=4,
                   alpha=0.7, label=feature_name)
    
    # Set geographic extent
    ax.set_extent([-120, -68, 30, 50], crs=cartopy.crs.PlateCarree())
    ax.set_title(f'GNSS TEC vs Magnetometer Network: Aurora event {event_start} - {event_end}', 
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='lower left')
    fig.show()

else:
    # Fallback: simple scatter plot without cartopy
    fig = matplotlib.pyplot.figure()
    ax = matplotlib.pyplot.subplot(1,1,1)

    # Plot 20 minute TEC scatter
    tecmap = ax.scatter(tec_lon, tec_lat, c=tec, cmap='jet',vmin=0.0,vmax=60.0, label='GNSS TEC')
    # Add the TEC color legend
    fig.colorbar(tecmap, label='TECu', shrink=0.4)
    
    for station_name, station_info in DASI_STATIONS.items():
        ax.scatter(station_info['longitude'], station_info['latitude'],
                   s=200, marker='s', color=colors[station_name],
                   edgecolors='black', linewidths=1.5,
                   label=station_name, zorder=5)
    
    for feature_name, feature in AURORAL_FEATURES.items():
        marker = feature['marker_style']['marker']
        size = feature['marker_style']['size']
        color = feature['marker_style']['color']
        
        ax.scatter(feature['lon'], feature['lat'],
                   s=size, marker=marker, color=color,
                   edgecolors='black', linewidths=1,
                   label=feature_name, alpha=0.7, zorder=4)
    
    ax.set_xlabel('Longitude (°W)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Latitude (°N)', fontsize=11, fontweight='bold')
    ax.set_title(f'GNSS TEC vs Magnetometer Network: Aurora event {event_start} - {event_end}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=8, loc='lower left')
    ax.set_xlim(-120, -68)
    ax.set_ylim(35, 50)
    fig.show()

/var/folders/t7/8t6qs11170n_htmckmyj34540000gn/T/ipykernel_44394/1643295145.py:82: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()


In [ ]:
# use the following helper function to use HAPI to query for Madrigal magnetometer data for this event
def get_magnetometer_df_from_hapi(kinst, kindat):
    """
    Query the Madrigal-HAPI server for magnetometer data and return as Pandas dataframe.

    Parameters:
    -----------
    kinst : int
        Instrument code for DASI station
    kindat : int
        Kind of data code for DASI station

    Returns:
    --------
    pandas.DataFrame or None
        DataFrame with columns: time, H, D, Z, and derived perturbations
        Returns None if query fails
    """
    server = f"{MADRIGAL_URL}hapi" # Madrigal-HAPI server name
    #requested_parms = "bd_nt,be_nt,bn_nt" # parameter request string per HAPI spec
    requested_parms = ""
    dataset = str(kinst) + "_" + str(kindat) # Madrigal-HAPI dataset ID: kinst_kindat
    data, meta = hapi(server, dataset, requested_parms, f"{EVENT_DATE}T{EVENT_START_UT}", f"{EVENT_DATE}T{EVENT_END_UT}")
    
    # use pandas to parse result file
    df = pandas.DataFrame(data)
    df["Time"] = df["Time"].apply(lambda x: x.decode('utf8'))
    df["Time"] = pandas.to_datetime(df["Time"], utc=True)
    df_new = df.rename(columns={"bn_nt": "H", "be_nt": "D", "bd_nt":"Z"})
    
    return(df_new)

In [ ]:
# use the following helper functions to use the madrigalWeb API to query for magnetometer data for this event
def query_magnetometer_data_madrigal(kinst, start_time, end_time, station_name):
    """
    Query magnetometer data from Madrigal database for a specific instrument
    and time range.
    
    Parameters:
    -----------
    kinst : int
        Instrument code for DASI station
    start_time : datetime
        Start time for query (UTC)
    end_time : datetime
        End time for query (UTC)
    station_name : str
        Human-readable station name (for logging)
    
    Returns:
    --------
    pandas.DataFrame or None
        DataFrame with columns: time, H, D, Z, and derived perturbations
        Returns None if query fails
    
    Notes:
    ------
    The Madrigal API retrieves data in HDF5 format. Common magnetic field
    components are:
      - Bn (or H): Northward magnetic field component
      - Be (or D): Eastward magnetic field component
      - Bd (or Z): Vertical (downward) magnetic field component
    """
    
    print(f"\nQuerying {station_name} (kinst={kinst})...")
    print(f"  Time range: {start_time.isoformat()} to {end_time.isoformat()}")
    print(f"  Duration: {(end_time - start_time).total_seconds() / 60:.1f} minutes")
    
    try:
        # Step 1: Find available experiments/files in Madrigal
        madrigal_db = madrigalWeb.madrigalWeb.MadrigalData(MADRIGAL_URL)
        print(f"  → Searching for data files...")
        
        experiments = madrigal_db.getExperiments(
            code=kinst,
            startyear=start_time.year,
            startmonth=start_time.month,
            startday=start_time.day,
            starthour=start_time.hour,
            startmin=start_time.minute,
            startsec=start_time.second,
            endyear=end_time.year,
            endmonth=end_time.month,
            endday=end_time.day,
            endhour=end_time.hour,
            endmin=end_time.minute,
            endsec=end_time.second
        )
        
        if not experiments:
            print(f"  ✗ No experiments found for this station on this date")
            return None
        
        print(f"  ✓ Found {len(experiments)} experiment(s)")
        
        # Step 2: Download matching files
        exp_ids = [e.id for e in experiments]
        
        files = []
        for exp_id in exp_ids:
            files += madrigal_db.getExperimentFiles(exp_id)
        
        if not files:
            print(f"  ✗ No files found in range {start_time} - {end_time}")
            return None

        print(f"  ✓ Retrieved {len(files)} files ")
        
        # Step 3: Read data from HDF5 file
        print(f"  → Reading HDF5 data...")
        
        # *UNVERIFIED* user ID info
        user_fullname = 'CEDAR Example'
        user_email = 'example@gmail.com'
        user_affiliation= 'CEDAR/GEM tutorial day 2026'
        
        # Download file
        for this_file in files:
            try:
                downloadedFile = f"/tmp/{os.path.basename(this_file.name)}"
                madrigal_db.downloadFile(this_file.name, downloadedFile, user_fullname, user_email, user_affiliation, format="hdf5")
                return(parse_madrigal_magnetometer_hdf5(downloadedFile, start_time, end_time))
            except Exception as e:
                traceback.print_exc()
                print(f"  ✗ Error downloading file: {e}")
                return None
        
    except Exception as e:
        print(f"  ✗ Unexpected error: {e}")
        traceback.print_exc()
        return None
    
def parse_madrigal_magnetometer_hdf5(this_file, start_time, end_time):
    # Parse HDF5
    try:
        with h5py.File(this_file, 'r') as f:
            # List available datasets
            print(f"  ✓ File opened. Available datasets: {list(f['Data/Table Layout'].dtype.names)}")
                    
            # Extract data
            # Note: Dataset names vary; common names are Bt, Be, Bz or H, D, Z
            if ('bn_nt' in f['Data/Table Layout'].dtype.names):
                h_data = f['Data/Table Layout']['bn_nt']
                d_data = f['Data/Table Layout']['be_nt']
                z_data = f['Data/Table Layout']['bd_nt']
            else:
                print(f"  ✗ Could not find magnetic field components")
                print(f"    Available: {list(f['Data/Table Layout'].dtype.names)}")
                return None
                    
            # Get time array
            if 'ut1_unix' in f['Data/Table Layout'].dtype.names:
                time_data = f['Data/Table Layout']['ut1_unix']
            else:
                print(f"  ✗ Could not find time data")
                return None
            
    except Exception as e:
        traceback.print_exc()
        print(f"  ✗ Error reading HDF5: {e}")
        return None
            
    # Convert Unix timestamps to datetime
    timestamps = pandas.to_datetime(time_data, unit='s', utc=True)
            
    # Create DataFrame
    df = pandas.DataFrame({
        'Time': timestamps,
        'H': h_data,
        'D': d_data,
        'Z': z_data
    })
            
    # Filter to requested time range (account for potential file boundaries)
    df = df[(df['Time'] >= start_time) & (df['Time'] <= end_time)].copy()
            
    if len(df) == 0:
        print(f"  ✗ No data in requested time range")
        return None
            
    return df

In [ ]:
# retrieve magnetometer data

mag_data = {}

for station_name, station_info in DASI_STATIONS.items():
    if USE_HAPI:
        df = get_magnetometer_df_from_hapi(station_info['kinst'], station_info['kindat'])
    else:
        df = query_magnetometer_data_madrigal(
            station_info['kinst'],
            event_start,
            event_end,
            station_name
        )
    
    if df is not None:
        mag_data[station_name] = df
        print(f"✓ {station_name}: Successfully retrieved")

In [ ]:
# calculate perterbations and compute data statistics
statistics = {}
baseline_start = event_start - datetime.timedelta(minutes=5)
baseline_end = event_start
breakup_peak = datetime.datetime.strptime(f"{EVENT_DATE} {BREAKUP_PEAK_UT}", "%Y-%m-%d %H:%M:%S%z")

for station_name, df in mag_data.items():
    baseline_mask = (df['Time'] >= baseline_start) & (df['Time'] < baseline_end)
            
    if len(df[baseline_mask]) > 0:
        H_baseline = df.loc[baseline_mask, 'H'].median()
        D_baseline = df.loc[baseline_mask, 'D'].median()
        Z_baseline = df.loc[baseline_mask, 'Z'].median()
    else:
        # Use overall median if baseline window not available
        H_baseline = df['H'].median()
        D_baseline = df['D'].median()
        Z_baseline = df['Z'].median()
            
    # Calculate perturbations
    df['dH'] = df['H'] - H_baseline
    df['dD'] = df['D'] - D_baseline
    df['dZ'] = df['Z'] - Z_baseline
    df['B_total'] = numpy.sqrt(df['H']**2 + df['D']**2 + df['Z']**2)
    
    # Find time of maximum H-component perturbation (minimum = most negative)
    idx_peak_dH = df['dH'].idxmin()
    time_peak_dH = df.loc[idx_peak_dH, 'Time']
    
    # Compile statistics
    stats = {
        'N_records': len(df),
        'Peak_dH': df['dH'].min(),
        'Peak_dD': df['dD'].max(),
        'Peak_dZ_abs': df['dZ'].abs().max(),
        'Time_Peak_dH': time_peak_dH,
        'dH_rate_of_change': (df['dH'].iloc[-1] - df['dH'].iloc[0]) / len(df),
        'Mean_dH': df['dH'].mean(),
        'Std_dH': df['dH'].std(),
    }
    
    statistics[station_name] = stats

In [ ]:
# Generate magnetometer plots
# PLOT: H-Component (Westward Electrojet)
# ========================================

ax = matplotlib.pyplot.subplot(1, 1, 1)

for station_name, df in mag_data.items():
    ax.plot(df['Time'], df['dH'], 
            label=station_name, 
            color=colors[station_name],
            linewidth=1.8, 
            alpha=0.85)

ax.axvline(breakup_peak, color='orange', linestyle='--', linewidth=2.5, 
           label='Breakup Peak', zorder=5)
ax.axhline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)

ax.set_xlabel('Time (UT)', fontsize=11, fontweight='bold')
ax.set_ylabel('ΔH (nT)', fontsize=11, fontweight='bold')
ax.set_title('H-Component: Westward Electrojet', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.25, linestyle='--')
ax.legend(fontsize=9, loc='lower left')
ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter('%H:%M:%S'))
matplotlib.pyplot.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

In [ ]:
# PLOT: D-Component (Eastward Surge)
# ========================================
ax = matplotlib.pyplot.subplot(1, 1, 1)

for station_name, df in mag_data.items():
    ax.plot(df['Time'], df['dD'], 
            label=station_name, 
            color=colors[station_name],
            linewidth=1.8, 
            alpha=0.85)

ax.axvline(breakup_peak, color='orange', linestyle='--', linewidth=2.5)
ax.axhline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)

ax.set_xlabel('Time (UT)', fontsize=11, fontweight='bold')
ax.set_ylabel('ΔD (nT)', fontsize=11, fontweight='bold')
ax.set_title('D-Component: Eastward Surge', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.25, linestyle='--')
ax.legend(fontsize=9, loc='upper left')
ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter('%H:%M:%S'))
matplotlib.pyplot.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

In [ ]:
# PLOT: Z-Component (Vertical Perturbation)
# ========================================
ax = matplotlib.pyplot.subplot(1, 1, 1)

for station_name, df in mag_data.items():
    ax.plot(df['Time'], df['dZ'], 
            label=station_name, 
            color=colors[station_name],
            linewidth=1.8, 
            alpha=0.85)

ax.axvline(breakup_peak, color='orange', linestyle='--', linewidth=2.5)
ax.axhline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)

ax.set_xlabel('Time (UT)', fontsize=11, fontweight='bold')
ax.set_ylabel('ΔZ (nT)', fontsize=11, fontweight='bold')
ax.set_title('Z-Component: Overhead Signature', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.25, linestyle='--')
ax.legend(fontsize=9, loc='upper right')
ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter('%H:%M:%S'))
matplotlib.pyplot.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

In [ ]:
# PLOT: Total Field Magnitude
# ========================================
ax = matplotlib.pyplot.subplot(1, 1, 1)

for station_name, df in mag_data.items():
    ax.plot(df['Time'], df['B_total'], 
            label=station_name, 
            color=colors[station_name],
            linewidth=1.8, 
            alpha=0.85)

ax.axvline(breakup_peak, color='orange', linestyle='--', linewidth=2.5)

ax.set_xlabel('Time (UT)', fontsize=11, fontweight='bold')
ax.set_ylabel('|B| (nT)', fontsize=11, fontweight='bold')
ax.set_title('Total Magnetic Field Magnitude', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.25, linestyle='--')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter('%H:%M:%S'))
matplotlib.pyplot.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

In [ ]:
# PLOT: H-Component Rate of Change
# ========================================
ax = matplotlib.pyplot.subplot(1,1,1)

for station_name, df in mag_data.items():
    # Calculate dH/dt
    dH_dt = df['dH'].diff() / (df['Time'].diff().dt.total_seconds())
    
    ax.plot(df['Time'][1:], dH_dt[1:], 
            label=station_name, 
            color=colors[station_name],
            linewidth=1.5, 
            alpha=0.85)

ax.axvline(breakup_peak, color='orange', linestyle='--', linewidth=2.5)
ax.axhline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)

ax.set_xlabel('Time (UT)', fontsize=11, fontweight='bold')
ax.set_ylabel('dH/dt (nT/s)', fontsize=11, fontweight='bold')
ax.set_title('H-Component Rate of Change', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.25, linestyle='--')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter('%H:%M:%S'))
matplotlib.pyplot.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')